# Precio de vivienda estimado por AGEB

Sesión 23c de Mapo. ¿Se puede dar, para cada AGEB urbana de la zona metropolitana de Guadalajara, un **rango** de valor de suelo con una cobertura medida, en vez de un número solo? Diseño en `docs/producto/precio-estimado.md`.

- **Objetivo:** `log(medv)`, la mediana del valor catastral de suelo de zona dentro de la AGEB ($/m²). Guadalajara usa la tabla de 2020; Zapopan, Tonalá e Ixtlahuacán, la de 2026. El efecto fijo por municipio absorbe los años distintos.
- **Rango:** predicción conformal partida, con 80 % de cobertura nominal, calibrada por bloques espaciales de 3 km.
- **Lo que no dice:** el rango cubre el catastro, no el precio de mercado. Mientras no haya precios reales para validar, la escala de mercado queda fuera.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from precio import conformal, datos, graficas, modelos, salida

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.precision", 3)

RAIZ = Path("../..")
DATASET = RAIZ / "mapo_core/data/vivienda_gdl_v7.zip"
SALIDA = RAIZ / "mapo_core/data/precio_estimado/precio_ageb_v1.csv"
VERSION = "v1-2026-11-20"
ALFA = 0.2  # cobertura nominal del 80 %
RASGOS = datos.RASGOS

## $\color{#dda}{\text{1. Datos}}$

2,068 AGEB urbanas de diez municipios. Solo cuatro tienen tabla de valores ubicada; las demás AGEB de esos cuatro son las que el modelo tiene que estimar.

In [2]:
v = datos.cargar(DATASET)
v["bloque"] = datos.bloques(v)
base = v.dropna(subset=["y"]).reset_index(drop=True)
v.assign(con_catastro=v["medv"].notna()).groupby("municipio").agg(
    agebs=("cvegeo", "size"), con_catastro=("con_catastro", "sum"), anio_tabla=("medv_anio", "first"))

,agebs,con_catastro,anio_tabla
municipio,,,
Acatlán de Juárez,31,0,NaN
El Salto,127,0,NaN
Guadalajara,442,415,2020.0
Ixtlahuacán de los Membrillos,37,23,2026.0
Juanacatlán,18,0,NaN
San Pedro Tlaquepaque,234,0,NaN
Tlajomulco de Zúñiga,377,0,NaN
Tonalá,228,152,2026.0
Zapopan,529,302,2026.0


In [3]:
print(f"{len(base)} AGEB con catastro en {base['bloque'].nunique()} bloques de 3 km")
graficas.distribucion(base)

892 AGEB con catastro en 85 bloques de 3 km


Zapopan, Tonalá e Ixtlahuacán salen de OCR sobre escaneos. Debajo de $1,000/m² el OCR confunde valores con números de manzana (`docs/producto/dataset-vivienda.md`): ahí están las colas bajas. No todas son error: en Zapopan las AGEB con menos de $1,000 tienen escolaridad mediana de 8.7 años contra 12.5 de las demás. Se dejan, y la sección 5 mide qué pasa sin ellas.

In [4]:
base.assign(menor_1000=base["medv"] < 1000).groupby("municipio").agg(
    agebs=("cvegeo", "size"), menor_1000=("menor_1000", "sum"))

,agebs,menor_1000
municipio,,
Guadalajara,415,0
Ixtlahuacán de los Membrillos,23,7
Tonalá,152,25
Zapopan,302,51


## $\color{#dda}{\text{2. Forma: qué modelo ordena mejor}}$

Cuatro candidatos con los mismos 44 rasgos de nivel AGEB y el efecto fijo por municipio. Cada AGEB se predice con un modelo que no vio su bloque de 3 km (cinco particiones). La SVR y la ridge buscan sus hiperparámetros dentro de cada partición, también por bloques.

In [5]:
X, y, g = base[RASGOS + ["municipio"]], base["y"], base["bloque"]
predicciones, tablas = {}, []
for nombre, modelo in modelos.catalogo(RASGOS).items():
    predicciones[nombre] = modelos.fuera_de_muestra(modelo, X, y, g)
    tablas.append(modelos.metricas(y, predicciones[nombre], base["municipio"]).assign(modelo=nombre))
comparacion = pd.concat(tablas)
comparacion.pivot(index="municipio", columns="modelo", values=["rmse_log", "spearman"])

rmse_log                      spearman         \
modelo                        Boosting Bosque  Ridge    SVR Boosting Bosque   
municipio                                                                     
Guadalajara                      0.237  0.196  0.217  0.210    0.773  0.823   
Ixtlahuacán de los Membrillos    1.019  0.862  0.890  0.956   -0.355  0.119   
Todos                            0.678  0.622  0.626  0.636    0.653  0.720   
Tonalá                           0.777  0.733  0.738  0.754    0.380  0.452   
Zapopan                          0.948  0.873  0.872  0.883    0.549  0.628   

                                             
modelo                         Ridge    SVR  
municipio                                    
Guadalajara                    0.814  0.799  
Ixtlahuacán de los Membrillos  0.135  0.038  
Todos                          0.729  0.735  
Tonalá                         0.481  0.460  
Zapopan                        0.648  0.639

In [6]:
graficas.comparacion(comparacion)

In [7]:
graficas.observado_contra_predicho(y, predicciones["Bosque"], base["municipio"],
                                   "Bosque aleatorio: catastro contra predicho fuera de muestra")

## $\color{#dda}{\text{3. Transferencia a un municipio sin tabla}}$

Si el orden predicho aguantara en un municipio entero que el modelo nunca vio, se podría dar forma a Tlaquepaque o Tlajomulco. Aun así faltaría el nivel: sin tabla no hay efecto fijo. Se deja fuera un municipio a la vez y se mide el Spearman dentro de él.

In [8]:
cat = modelos.catalogo(RASGOS)
pd.concat([modelos.transferencia(cat[n], base, RASGOS).assign(modelo=n) for n in ["Ridge", "SVR", "Bosque"]]
          ).pivot(index="municipio", columns="modelo", values="spearman")

modelo,Bosque,Ridge,SVR
municipio,,,
Guadalajara,0.789,0.808,0.779
Ixtlahuacán de los Membrillos,-0.045,-0.051,0.130
Tonalá,0.417,0.479,0.473
Zapopan,0.619,0.648,0.665


## $\color{#dda}{\text{4. Rango conformal}}$

Cada método se evalúa en AGEB que ni el ajuste ni la calibración vieron: afuera, cinco particiones por bloques; adentro, el resto se parte en ajuste (70 %) y calibración (30 %), también por bloques.

- **Normalizado:** |y − ŷ| / σ̂, con σ̂ de un bosque que aprende el tamaño del error fuera de muestra. Rango ŷ ± q·σ̂.
- **CQR:** bordes de boosting de cuantiles en 10 % y 90 %, abiertos o cerrados por la calibración.
- **Al azar:** el mismo normalizado partiendo AGEB sueltas, para ver cuánto angosta el rango ignorar a las vecinas.

In [9]:
fabricas = {
    "Normalizado, bosque": (lambda: conformal.Normalizado(cat["Bosque"], RASGOS), True),
    "Normalizado, SVR": (lambda: conformal.Normalizado(cat["SVR"], RASGOS), True),
    "CQR": (lambda: conformal.CQR(RASGOS, ALFA), True),
    "Normalizado, bosque, al azar": (lambda: conformal.Normalizado(cat["Bosque"], RASGOS), False),
}
rangos = {n: conformal.evaluar(f, base, RASGOS, ALFA, espacial=e) for n, (f, e) in fabricas.items()}
cobertura = pd.concat([conformal.resumen(r).assign(metodo=n) for n, r in rangos.items()])
cobertura.pivot(index="municipio", columns="metodo", values=["cobertura", "alto_entre_bajo_mediana"])

cobertura                                       \
metodo                              CQR Normalizado, SVR Normalizado, bosque   
municipio                                                                      
Guadalajara                       0.880            0.793               0.786   
Ixtlahuacán de los Membrillos     0.652            0.696               0.826   
Todos                             0.789            0.774               0.796   
Tonalá                            0.796            0.783               0.836   
Zapopan                           0.672            0.748               0.788   

                                                            \
metodo                        Normalizado, bosque, al azar   
municipio                                                    
Guadalajara                                          0.805   
Ixtlahuacán de los Membrillos                        0.826   
Todos                                                0.800   
Tonalá                                               0.822   
Zapopan                                              0.781   

                              alto_entre_bajo_mediana                   \
metodo                                            CQR Normalizado, SVR   
municipio                                                                
Guadalajara                                     1.871            1.517   
Ixtlahuacán de los Membrillos                   5.942            6.981   
Todos                                           2.706            2.490   
Tonalá                                          4.050            4.153   
Zapopan                                         4.295            4.215   

                                                                                
metodo                        Normalizado, bosque Normalizado, bosque, al azar  
municipio                                                                       
Guadalajara                                 1.488                        1.424  
Ixtlahuacán de los Membrillos              10.262                        9.428  
Todos                                       3.124                        2.689  
Tonalá                                      5.165                        3.799  
Zapopan                                     5.196                        4.683

In [10]:
graficas.cobertura(cobertura, 1 - ALFA)

In [11]:
graficas.mapa_ancho(rangos["Normalizado, bosque"], base)

## $\color{#dda}{\text{5. Sensibilidad: sin valores menores a \$1,000}}$

Lo mismo, quitando las AGEB de municipios con OCR cuyo valor es menor a $1,000/m². Si el ancho baja mucho, el ruido del OCR pesa más que el modelo.

In [12]:
limpia = base[~((base["medv"] < 1000) & (base["municipio"] != "Guadalajara"))].reset_index(drop=True)
r_limpia = conformal.evaluar(lambda: conformal.Normalizado(cat["Bosque"], RASGOS), limpia, RASGOS, ALFA)
pd.concat([conformal.resumen(rangos["Normalizado, bosque"]).assign(datos="todos"),
           conformal.resumen(r_limpia).assign(datos="sin < 1,000")]
          ).pivot(index="municipio", columns="datos", values=["agebs", "cobertura", "alto_entre_bajo_mediana"])

agebs          cobertura         \
datos                         sin < 1,000  todos sin < 1,000  todos   
municipio                                                             
Guadalajara                         415.0  415.0       0.839  0.786   
Ixtlahuacán de los Membrillos        16.0   23.0       0.812  0.826   
Todos                               809.0  892.0       0.812  0.796   
Tonalá                              127.0  152.0       0.780  0.836   
Zapopan                             251.0  302.0       0.785  0.788   

                              alto_entre_bajo_mediana          
datos                                     sin < 1,000   todos  
municipio                                                      
Guadalajara                                     1.542   1.488  
Ixtlahuacán de los Membrillos                   2.339  10.262  
Todos                                           1.953   3.124  
Tonalá                                          2.261   5.165  
Zapopan                                         2.725   5.196

## $\color{#dda}{\text{6. Tabla para Mapo}}$

Con el método elegido se ajusta con todas las AGEB con catastro (70 % ajuste, 30 % calibración, por bloques). Cada AGEB sin catastro de esos cuatro municipios recibe su rango y la cobertura medida en su municipio en la sección 4. Las de municipios sin tabla quedan `sin_base`.

In [13]:
elegido = "Normalizado, bosque"
cob_mun = conformal.resumen(rangos[elegido]).set_index("municipio")["cobertura"]
t = salida.tabla(v, fabricas[elegido][0], RASGOS, ALFA, cob_mun, VERSION)
SALIDA.parent.mkdir(parents=True, exist_ok=True)
t.to_csv(SALIDA, index=False)
t.groupby(["municipio", "fuente"]).size().unstack(fill_value=0)

fuente,catastro,modelo,sin_base
municipio,,,
Acatlán de Juárez,0,0,31
El Salto,0,0,127
Guadalajara,415,27,0
Ixtlahuacán de los Membrillos,23,14,0
Juanacatlán,0,0,18
San Pedro Tlaquepaque,0,0,234
Tlajomulco de Zúñiga,0,0,377
Tonalá,152,76,0
Zapopan,302,227,0


In [14]:
t[t["fuente"] == "modelo"].assign(alto_entre_bajo=lambda d: d["rango_alto"] / d["rango_bajo"]).groupby(
    "municipio")[["rango_bajo", "rango_alto", "alto_entre_bajo", "cobertura"]].median()

,rango_bajo,rango_alto,alto_entre_bajo,cobertura
municipio,,,,
Guadalajara,2660.0,4210.0,1.553,0.786
Ixtlahuacán de los Membrillos,175.0,7335.0,50.261,0.826
Tonalá,365.0,6230.0,17.177,0.836
Zapopan,390.0,8250.0,22.931,0.788


## $\color{#dda}{\text{7. Qué queda}}$

**Lo que sale (corrida del 2026-11-20):**

- **Forma.** Los cuatro modelos empatan casi: RMSE en log de 0.62 a 0.68 fuera de muestra por bloques. Guadalajara ordena bien (Spearman 0.77 a 0.82, error típico de 20 %); Zapopan, 0.55 a 0.65; Tonalá, 0.38 a 0.48; Ixtlahuacán, nada (23 AGEB). El bosque gana por poco y es el elegido; la SVR queda a la par, no mejor.
- **Transferencia.** Con un municipio entero fuera, el orden aguanta en Guadalajara (0.78 a 0.81) y en Zapopan (0.62 a 0.67), menos en Tonalá (0.42 a 0.48) y nada en Ixtlahuacán. Aun así, a un municipio sin tabla le faltaría el nivel: quedan `sin_base`.
- **Rango.** Normalizado con bosque cubre 80 % en total y 79 a 84 % por municipio. CQR se queda en 67 % en Zapopan. Partir al azar da rangos 14 % más angostos en total y 26 % en Tonalá: es lo que se promete de más cuando la calibración tiene vecinas de las AGEB evaluadas.
- **Ancho.** En Guadalajara el alto es 1.5 veces el bajo: un rango útil. En Zapopan y Tonalá, unas 5 veces en las AGEB evaluadas y más de 20 en las que se estiman, que están en la periferia. Sin las etiquetas menores a $1,000 de los municipios con OCR, el ancho baja a la mitad con la misma cobertura: el ruido del OCR pesa más que el modelo.

**Límites:**

- El rango cubre el catastro, no el mercado. Sin precios reales no hay escala de mercado.
- La cobertura se mide en AGEB con catastro. Las que se estiman están más en la orilla, donde hubo menos hojas ubicadas; nada garantiza que se parezcan.
- Las AGEB con catastro de Guadalajara usan la tabla de 2020 y las de los otros tres municipios la de 2026.

**Siguiente:** revisar a mano una muestra de valores menores a $1,000 en los escaneos de Zapopan y Tonalá. Si la mayoría es número de manzana, subir el filtro del pipeline a $1,000 y rehacer el dataset y esta tabla.